# 🔬 Advanced Semantic Research Paper Search Engine

An end-to-end **NLP information-retrieval system** built on top of the ML-ArXiv-Papers dataset.

**Original features (kept):**
- Sentence-Transformer embeddings (`all-MiniLM-L6-v2`)
- FAISS vector similarity search
- Abstractive summarization (BART)
- Keyword extraction (KeyBERT)

**New features added in this version:**
1. 🗄️ **Embedding & index caching** — compute once, reload instantly next time
2. 🔎 **BM25 keyword search** and **Hybrid Search** (semantic + lexical fusion)
3. 🎯 **Cross-Encoder re-ranking** for higher precision top results
4. ✍️ **Query expansion** using WordNet synonyms
5. 📎 **"More like this" paper recommendation**
6. 🧭 **Topic clustering (KMeans)** with 2D visualization (PCA) and word clouds per cluster
7. 🧩 A single reusable **`PaperSearchEngine`** class wrapping the whole pipeline
8. 💾 **Export search results** to CSV / JSON
9. 🖱️ **Interactive search UI** with `ipywidgets` (query box, filters, toggles)
10. 📊 **Search history & analytics**

> This notebook is designed to run on Google Colab (GPU recommended for embeddings & summarization).


## 1. Setup & Installation

In [ ]:
# Core dependencies (original)
!pip install -q datasets sentence-transformers faiss-cpu transformers==4.46.3 keybert

# New dependencies for the advanced features
!pip install -q rank_bm25 scikit-learn plotly wordcloud ipywidgets nltk

In [ ]:
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

## 2. Imports & Configuration

In [ ]:
import os
import json
import re
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import faiss
from rank_bm25 import BM25Okapi
from transformers import pipeline
from keybert import KeyBERT
from wordcloud import WordCloud
from nltk.corpus import wordnet

import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

warnings.filterwarnings("ignore")

In [ ]:
# ---- Global configuration ----
CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
SUMMARIZER_MODEL_NAME = "facebook/bart-large-cnn"

EMBEDDING_DIM = 384          # matches all-MiniLM-L6-v2
NUM_PAPERS = 15000           # subset size (same as original project)
RANDOM_STATE = 42

EMBEDDINGS_PATH = os.path.join(CACHE_DIR, "embeddings.npy")
FAISS_INDEX_PATH = os.path.join(CACHE_DIR, "faiss.index")
DATAFRAME_PATH = os.path.join(CACHE_DIR, "papers.parquet")

## 3. Data Loading

Same source dataset as the original project (`CShorten/ML-ArXiv-Papers`), now wrapped in a
function with a **local parquet cache** so re-running the notebook doesn't require
re-downloading and re-processing the dataset every time.

In [ ]:
def load_papers(num_papers=NUM_PAPERS, force_reload=False):
    """Load the ML-ArXiv-Papers dataset, clean it, and cache to disk as parquet."""
    if os.path.exists(DATAFRAME_PATH) and not force_reload:
        print(f"Loading cached dataframe from {DATAFRAME_PATH}")
        return pd.read_parquet(DATAFRAME_PATH)

    print("Downloading dataset from Hugging Face Hub...")
    dataset = load_dataset("CShorten/ML-ArXiv-Papers", split="train")
    df = pd.DataFrame(dataset)
    df = df[["title", "abstract"]]
    df = df.head(num_papers).reset_index(drop=True)
    df = df.dropna(subset=["title", "abstract"]).reset_index(drop=True)

    df["paper_text"] = clean_text_series(df["title"] + " " + df["abstract"])

    df.to_parquet(DATAFRAME_PATH)
    print(f"Cached {len(df)} papers to {DATAFRAME_PATH}")
    return df


def clean_text_series(series):
    """Vectorized text cleaning: strip newlines/extra whitespace/control chars."""
    series = series.str.replace(r"\s+", " ", regex=True)
    series = series.str.strip()
    return series

In [ ]:
df = load_papers()
print(df.shape)
df.head()

## 4. Embedding Generation (NEW: disk caching)

Embeddings are the most expensive step. We now **cache them to disk** (`cache/embeddings.npy`)
so subsequent runs load in milliseconds instead of re-encoding 15,000 papers.

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

def build_embeddings(df, force_reload=False):
    if os.path.exists(EMBEDDINGS_PATH) and not force_reload:
        print(f"Loading cached embeddings from {EMBEDDINGS_PATH}")
        embeddings = np.load(EMBEDDINGS_PATH)
    else:
        print("Encoding papers... this may take a while.")
        embeddings = embedding_model.encode(
            df["paper_text"].tolist(),
            batch_size=32,
            show_progress_bar=True,
            convert_to_numpy=True,
        )
        np.save(EMBEDDINGS_PATH, embeddings)
        print(f"Saved embeddings to {EMBEDDINGS_PATH}")

    return embeddings.astype("float32")

embeddings = build_embeddings(df)
print(embeddings.shape, embeddings.dtype)

## 5. FAISS Vector Index (NEW: save/load index)

In [ ]:
def build_faiss_index(embeddings, force_reload=False):
    if os.path.exists(FAISS_INDEX_PATH) and not force_reload:
        print(f"Loading cached FAISS index from {FAISS_INDEX_PATH}")
        return faiss.read_index(FAISS_INDEX_PATH)

    normed = embeddings.copy()
    faiss.normalize_L2(normed)
    index = faiss.IndexFlatIP(normed.shape[1])
    index.add(normed)
    faiss.write_index(index, FAISS_INDEX_PATH)
    print(f"Built and cached FAISS index with {index.ntotal} vectors")
    return index

faiss_index = build_faiss_index(embeddings)
print("Total vectors indexed:", faiss_index.ntotal)

## 6. BM25 Keyword Search (NEW)

Semantic search is great for meaning but can miss exact keyword/acronym matches
(e.g. "BERT", "GAN"). BM25 is a classic lexical ranking algorithm that complements it.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-zA-Z0-9]+", text.lower())

print("Building BM25 index...")
tokenized_corpus = [tokenize(t) for t in df["paper_text"].tolist()]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index ready.")

In [ ]:
def bm25_search(query, k=5):
    scores = bm25.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top_idx]

bm25_search("deep learning for medical image analysis", k=5)

## 7. Hybrid Search (NEW): Semantic + BM25 fusion

Both score sets are min-max normalized and combined with a tunable weight `alpha`
(`alpha=1` → pure semantic, `alpha=0` → pure keyword/BM25).

In [ ]:
def _min_max_normalize(scores):
    scores = np.array(scores, dtype="float64")
    if scores.max() - scores.min() < 1e-9:
        return np.zeros_like(scores)
    return (scores - scores.min()) / (scores.max() - scores.min())


def hybrid_search(query, k=5, alpha=0.6, candidate_pool=50):
    """alpha: weight given to semantic similarity vs BM25 (0-1)."""
    # Semantic candidates
    q_emb = embedding_model.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)
    sem_scores, sem_idx = faiss_index.search(q_emb, candidate_pool)
    sem_scores, sem_idx = sem_scores[0], sem_idx[0]

    # BM25 candidates
    bm25_scores_full = bm25.get_scores(tokenize(query))

    # Union of candidate indices from both retrievers
    candidate_ids = set(sem_idx.tolist()) | set(
        np.argsort(bm25_scores_full)[::-1][:candidate_pool].tolist()
    )
    candidate_ids = list(candidate_ids)

    sem_map = dict(zip(sem_idx.tolist(), sem_scores.tolist()))
    combined = []
    sem_vals = [sem_map.get(i, 0.0) for i in candidate_ids]
    bm25_vals = [bm25_scores_full[i] for i in candidate_ids]

    sem_norm = _min_max_normalize(sem_vals)
    bm25_norm = _min_max_normalize(bm25_vals)

    for idx, s_norm, b_norm in zip(candidate_ids, sem_norm, bm25_norm):
        final_score = alpha * s_norm + (1 - alpha) * b_norm
        combined.append((idx, final_score))

    combined.sort(key=lambda x: x[1], reverse=True)
    return combined[:k]

hybrid_search("transformer models for time series forecasting", k=5)

## 8. Query Expansion (NEW): WordNet synonym augmentation

Expands short/ambiguous queries with synonyms before searching, which can improve recall
for queries that use different vocabulary than the papers.

In [ ]:
def expand_query(query, max_synonyms_per_word=2):
    words = query.split()
    expanded_terms = set(words)
    for w in words:
        synsets = wordnet.synsets(w)
        syns_added = 0
        for syn in synsets:
            for lemma in syn.lemmas():
                name = lemma.name().replace("_", " ")
                if name.lower() != w.lower() and syns_added < max_synonyms_per_word:
                    expanded_terms.add(name)
                    syns_added += 1
    return " ".join(expanded_terms)

print(expand_query("fast image classification"))

## 9. Cross-Encoder Re-Ranking (NEW)

Bi-encoder (SentenceTransformer) retrieval is fast but less precise. A **cross-encoder**
jointly encodes (query, passage) pairs for much more accurate relevance scoring — we use
it to re-rank the top candidates retrieved by hybrid search.

In [ ]:
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL_NAME)

def rerank(query, candidate_indices, top_k=5):
    pairs = [(query, df.iloc[idx]["abstract"]) for idx in candidate_indices]
    scores = cross_encoder.predict(pairs)
    reranked = sorted(zip(candidate_indices, scores), key=lambda x: x[1], reverse=True)
    return reranked[:top_k]

# Example: retrieve wide candidate pool with hybrid search, then re-rank precisely
_candidates = [idx for idx, _ in hybrid_search("medical image analysis with deep learning", k=20)]
rerank("medical image analysis with deep learning", _candidates, top_k=5)

## 10. Abstractive Summarization (kept from original, wrapped in a function)

In [ ]:
summarizer = pipeline("summarization", model=SUMMARIZER_MODEL_NAME)

def summarize_abstract(text, max_length=120, min_length=40):
    text = text[:4000]  # guard against overly long inputs
    result = summarizer(text, max_length=max_length, min_length=min_length, do_sample=False)
    return result[0]["summary_text"]

summarize_abstract(df.iloc[0]["abstract"])

## 11. Keyword Extraction (kept from original, KeyBERT)

In [ ]:
kw_model = KeyBERT(embedding_model)

def extract_keywords(text, top_n=8):
    return kw_model.extract_keywords(
        text, keyphrase_ngram_range=(1, 2), stop_words="english", top_n=top_n
    )

extract_keywords(df.iloc[0]["abstract"])

## 12. "More Like This" Paper Recommendation (NEW)

Given a paper by its index, find the most similar other papers in the corpus —
useful for "related work" style browsing.

In [ ]:
def recommend_similar_papers(paper_idx, k=5):
    query_vec = embeddings[paper_idx:paper_idx + 1].copy()
    faiss.normalize_L2(query_vec)
    scores, idxs = faiss_index.search(query_vec, k + 1)  # +1 to skip itself
    results = [(i, s) for s, i in zip(scores[0], idxs[0]) if i != paper_idx][:k]
    return results

for idx, score in recommend_similar_papers(0, k=5):
    print(f"[{score:.3f}] {df.iloc[idx]['title']}")

## 13. Topic Clustering & Visualization (NEW)

Cluster all papers with **KMeans** on their embeddings, project to 2D with **PCA**, and
visualize the topic landscape. Each cluster's top keywords are extracted from its centroid's
nearest papers, and a word cloud is generated per cluster.

In [ ]:
N_CLUSTERS = 12

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)
df["cluster"] = cluster_labels

pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords_2d = pca.fit_transform(embeddings)
df["pca_x"], df["pca_y"] = coords_2d[:, 0], coords_2d[:, 1]

plt.figure(figsize=(10, 7))
scatter = plt.scatter(df["pca_x"], df["pca_y"], c=df["cluster"], cmap="tab20", s=6, alpha=0.6)
plt.title("Paper Topic Clusters (PCA projection of embeddings)")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.colorbar(scatter, label="Cluster")
plt.show()

In [ ]:
def get_cluster_top_terms(cluster_id, top_n=10):
    cluster_texts = df.loc[df["cluster"] == cluster_id, "paper_text"]
    joined_text = " ".join(cluster_texts.sample(min(50, len(cluster_texts)), random_state=1))
    keywords = kw_model.extract_keywords(
        joined_text, keyphrase_ngram_range=(1, 2), stop_words="english", top_n=top_n
    )
    return keywords


def plot_cluster_wordcloud(cluster_id):
    terms = dict(get_cluster_top_terms(cluster_id, top_n=25))
    wc = WordCloud(width=600, height=350, background_color="white").generate_from_frequencies(terms)
    plt.figure(figsize=(8, 5))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Cluster {cluster_id} — top terms")
    plt.show()

plot_cluster_wordcloud(0)

## 14. Unified `PaperSearchEngine` Class (NEW)

Wraps the entire pipeline (embeddings, FAISS, BM25, hybrid search, re-ranking, summarization,
keyword extraction, recommendations, export, and search history) into a single reusable object.

In [ ]:
class PaperSearchEngine:
    def __init__(self, df, embeddings, faiss_index, bm25, embedding_model,
                 cross_encoder=None, summarizer=None, kw_model=None):
        self.df = df
        self.embeddings = embeddings
        self.faiss_index = faiss_index
        self.bm25 = bm25
        self.embedding_model = embedding_model
        self.cross_encoder = cross_encoder
        self.summarizer = summarizer
        self.kw_model = kw_model
        self.history = []  # search analytics log

    # ---------- Core search ----------
    def search(self, query, k=5, mode="hybrid", alpha=0.6, use_query_expansion=False,
               rerank_results=False, summarize=False, extract_kw=False):
        t0 = time.time()
        q = expand_query(query) if use_query_expansion else query

        if mode == "semantic":
            q_emb = self.embedding_model.encode([q]).astype("float32")
            faiss.normalize_L2(q_emb)
            scores, idxs = self.faiss_index.search(q_emb, k if not rerank_results else max(k, 20))
            results = list(zip(idxs[0].tolist(), scores[0].tolist()))
        elif mode == "bm25":
            results = bm25_search(q, k=k if not rerank_results else max(k, 20))
        else:  # hybrid
            results = hybrid_search(q, k=k if not rerank_results else max(k, 20), alpha=alpha)

        if rerank_results and self.cross_encoder is not None:
            candidate_ids = [idx for idx, _ in results]
            results = rerank(query, candidate_ids, top_k=k)
        else:
            results = results[:k]

        enriched = []
        for idx, score in results:
            row = self.df.iloc[int(idx)]
            entry = {
                "index": int(idx),
                "score": float(score),
                "title": row["title"],
                "abstract": row["abstract"],
            }
            if summarize and self.summarizer is not None:
                entry["summary"] = summarize_abstract(row["abstract"])
            if extract_kw and self.kw_model is not None:
                entry["keywords"] = extract_keywords(row["abstract"])
            enriched.append(entry)

        elapsed = time.time() - t0
        self.history.append({
            "timestamp": datetime.now().isoformat(),
            "query": query,
            "mode": mode,
            "k": k,
            "elapsed_sec": round(elapsed, 3),
            "result_titles": [r["title"] for r in enriched],
        })
        return enriched

    # ---------- Related features ----------
    def recommend(self, paper_idx, k=5):
        return recommend_similar_papers(paper_idx, k=k)

    def export_results(self, results, path="search_results.json", fmt="json"):
        if fmt == "json":
            with open(path, "w") as f:
                json.dump(results, f, indent=2)
        elif fmt == "csv":
            pd.DataFrame(results).to_csv(path, index=False)
        else:
            raise ValueError("fmt must be 'json' or 'csv'")
        print(f"Saved {len(results)} results to {path}")
        return path

    def get_history(self):
        return pd.DataFrame(self.history)

    def pretty_print(self, results):
        for r in results:
            print(f"[{r['score']:.3f}] {r['title']}")
            print(r["abstract"][:280] + "...")
            if "summary" in r:
                print("Summary:", r["summary"])
            if "keywords" in r:
                print("Keywords:", ", ".join(k for k, _ in r["keywords"]))
            print("-" * 100)


engine = PaperSearchEngine(
    df=df,
    embeddings=embeddings,
    faiss_index=faiss_index,
    bm25=bm25,
    embedding_model=embedding_model,
    cross_encoder=cross_encoder,
    summarizer=summarizer,
    kw_model=kw_model,
)
print("PaperSearchEngine ready.")

In [ ]:
results = engine.search(
    "deep learning for medical image analysis",
    k=5,
    mode="hybrid",
    rerank_results=True,
    summarize=True,
    extract_kw=True,
)
engine.pretty_print(results)

## 15. Export Results (NEW)

In [ ]:
engine.export_results(results, path="search_results.json", fmt="json")
engine.export_results(results, path="search_results.csv", fmt="csv")

## 16. Interactive Search UI (NEW: `ipywidgets`)

A simple point-and-click search interface — type a query, pick the search mode and options,
and view results without writing any code.

In [ ]:
query_box = widgets.Text(
    value="deep learning for medical image analysis",
    placeholder="Type your research query...",
    description="Query:",
    layout=widgets.Layout(width="600px"),
)
k_slider = widgets.IntSlider(value=5, min=1, max=15, description="Top K:")
mode_dropdown = widgets.Dropdown(
    options=["hybrid", "semantic", "bm25"], value="hybrid", description="Mode:"
)
rerank_checkbox = widgets.Checkbox(value=True, description="Cross-encoder re-rank")
summarize_checkbox = widgets.Checkbox(value=False, description="Summarize")
keywords_checkbox = widgets.Checkbox(value=False, description="Extract keywords")
search_button = widgets.Button(description="🔍 Search", button_style="primary")
output_area = widgets.Output()


def on_search_clicked(b):
    with output_area:
        clear_output()
        res = engine.search(
            query_box.value,
            k=k_slider.value,
            mode=mode_dropdown.value,
            rerank_results=rerank_checkbox.value,
            summarize=summarize_checkbox.value,
            extract_kw=keywords_checkbox.value,
        )
        for r in res:
            display(Markdown(f"### [{r['score']:.3f}] {r['title']}"))
            display(Markdown(r["abstract"][:400] + "..."))
            if "summary" in r:
                display(Markdown(f"**Summary:** {r['summary']}"))
            if "keywords" in r:
                kw_str = ", ".join(k for k, _ in r["keywords"])
                display(Markdown(f"**Keywords:** {kw_str}"))
            display(Markdown("---"))


search_button.on_click(on_search_clicked)

ui = widgets.VBox([
    query_box,
    widgets.HBox([k_slider, mode_dropdown]),
    widgets.HBox([rerank_checkbox, summarize_checkbox, keywords_checkbox]),
    search_button,
    output_area,
])
display(ui)

## 17. Search History & Analytics (NEW)

In [ ]:
history_df = engine.get_history()
history_df

In [ ]:
if len(history_df) > 0:
    plt.figure(figsize=(8, 4))
    plt.bar(range(len(history_df)), history_df["elapsed_sec"])
    plt.xticks(range(len(history_df)), history_df["query"], rotation=45, ha="right")
    plt.ylabel("Search latency (s)")
    plt.title("Search Latency per Query")
    plt.tight_layout()
    plt.show()

## 18. Summary

This notebook upgraded the original single-path semantic search project into a more complete
retrieval system:

| Area | Original | Advanced version |
|---|---|---|
| Retrieval | Semantic (FAISS) only | Semantic + BM25 + **Hybrid fusion** |
| Precision | Raw similarity ranking | + **Cross-encoder re-ranking** |
| Recall | Exact query only | + **WordNet query expansion** |
| Performance | Recomputed every run | + **Disk caching** for embeddings & index |
| Discovery | Manual browsing | + **"More like this" recommendations** |
| Corpus understanding | None | + **KMeans topic clustering** & word clouds |
| Usability | Function calls only | + **Interactive `ipywidgets` UI** |
| Ops | No persistence | + **JSON/CSV export**, **search history/analytics** |

Everything is exposed through the reusable `PaperSearchEngine` class, so it can be dropped
into another notebook, a script, or a small web app (e.g. Streamlit/Gradio) with minimal changes.
